# Adversarial attacks — `natquest` × `roberta-base`

Attacks **[namesarnav/natquest-roberta-base](https://huggingface.co/namesarnav/natquest-roberta-base)** on **every data point** of
`natquest` (13,412 rows, ~11 words each) with all 18 TextAttack recipes.

Output: one file per recipe on Drive, in `MyDrive/mdg/results/namesarnav_natquest__all/roberta-base/`
- `<recipe>.jsonl`: one line per example, with the original and perturbed text and both predictions
- `<recipe>_summary.json`: attack success rate, plus clean and attacked micro/macro F1

**Use a GPU runtime** (*Runtime → Change runtime type → T4 or better*). Then *Runtime → Run all*.

**If the session dies:** reconnect and *Run all* again. Every attacked example
is on Drive within 2 minutes, and the run resumes at the exact example where it
stopped. Local disk only ever holds the part in progress, so it stays far below
Colab's limit.

## 0. Settings

In [ ]:
DATASET = 'natquest'
MODEL   = 'namesarnav/natquest-roberta-base'
LABELS  = 'YES,NO'
CHUNK   = 200          # rows per part file on Drive

# Running this SAME notebook in several Colab sessions at once? Give each a
# different slice: '0/2' and '1/2', or '0/3', '1/3', '2/3'. Leave '0/1' otherwise.
WORKER  = '0/1'

QUERY_BUDGET        = None   # None = unlimited search, as in the original runs
EXAMPLE_TIMEOUT_MIN = 30     # an example still running after this is recorded as Timeout

DRIVE = '/content/drive/MyDrive/mdg'
LOCAL = '/content/mdg-local'

## 1. GPU

In [ ]:
import subprocess
try:
    out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                          '--format=csv,noheader,nounits'], capture_output=True, text=True).stdout
    gpu, vram = out.strip().split(', ')
    vram = int(vram)
except Exception:
    gpu, vram = None, 0
# Queries per forward pass. TextAttack's default of 32 leaves most of a GPU idle.
BATCH = 512 if vram >= 38000 else 256 if vram >= 20000 else 128 if vram >= 14000 else 64 if vram else None
print(f'GPU: {gpu or "NONE -- switch the runtime type to a GPU"}  ({vram} MB)  query batch: {BATCH}')

## 2. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(LOCAL, exist_ok=True)

## 3. Code and dependencies

In [ ]:
import os
if not os.path.isdir('/content/mdg/.git'):
    # GIT_LFS_SKIP_SMUDGE: clone code only. The data comes from Drive, or just
    # this dataset's files are pulled in the next cell.
    !GIT_LFS_SKIP_SMUDGE=1 git clone -q --depth 1 https://github.com/namesarnav/mdg.git /content/mdg
else:
    !cd /content/mdg && GIT_LFS_SKIP_SMUDGE=1 git pull -q --ff-only
%cd /content/mdg
!git log --oneline -1
!pip install -q "textattack==0.3.11" "transformers>=4.46,<5" accelerate sentencepiece "datasets>=3.0" "huggingface_hub<1.0" tensorflow_hub scikit-learn
!pip cache purge -q 2>/dev/null; true
!python3 colab/nltk_check.py

## 4. Data — every data point

`train + validation + test`, exact duplicates dropped. Built once from Git LFS
and kept in `MyDrive/mdg/data/`, so later sessions (and the other two models on
this dataset) copy it instead of pulling it again.

In [ ]:
import os, shutil
fname = f'namesarnav_{DATASET}__all.jsonl'
on_drive = f'{DRIVE}/data/{fname}'
local = f'/content/mdg/mdg/finetune/data/{fname}'
if os.path.exists(on_drive):
    shutil.copyfile(on_drive, local)
    print('copied from Drive:', on_drive)
else:
    !apt-get -qq install -y git-lfs > /dev/null
    !cd /content/mdg && git lfs install --local > /dev/null && git lfs pull --include="mdg/finetune/data/namesarnav_{DATASET}__*.jsonl"
    !cd /content/mdg && python3 colab/attack_all.py prepare --dataset {DATASET}
    os.makedirs(f'{DRIVE}/data', exist_ok=True)
    shutil.copyfile(local, on_drive)
    print('saved to Drive:', on_drive)
!wc -l {local}

## 5. Check every recipe

Builds each recipe and attacks 2 examples. This also downloads every recipe's
assets (embeddings, sentence encoders, language models), a few GB in all, to
local disk. A recipe that fails here would fail on every row, so this stops
the notebook instead of letting it burn the session.

In [ ]:
ARGS = (f'--dataset {DATASET} --model {MODEL} --labels {LABELS} --chunk {CHUNK} '
        f'--worker {WORKER} --drive {DRIVE} --local {LOCAL}')
if BATCH:               ARGS += f' --batch {BATCH}'
if QUERY_BUDGET:        ARGS += f' --query-budget {QUERY_BUDGET}'
if EXAMPLE_TIMEOUT_MIN: ARGS += f' --example-timeout-min {EXAMPLE_TIMEOUT_MIN}'
r = !cd /content/mdg && python3 colab/attack_all.py check {ARGS} 2>&1
print('\n'.join(r))
if any('[FAIL]' in l for l in r):
    raise SystemExit('A recipe failed its check -- see above. Not starting the run.')

## 6. Run

Recipes go cheapest first. Progress prints every 2 minutes, and every finished
recipe becomes one `.jsonl` + `_summary.json` on Drive. **Re-run this cell
(or Run all) to continue after a disconnect.**

In [ ]:
!cd /content/mdg && python3 -u colab/attack_all.py run {ARGS} 2>&1 | tee -a {LOCAL}/run.log | grep --line-buffered -v -E "^(WARNING|I0000|W0000)"

## 7. Progress (safe to run any time, from any session)

In [ ]:
!cd /content/mdg && python3 colab/attack_all.py status {ARGS}
!df -h /content | tail -1